# Module 5 Instructor Colab Notebook: Fundamentals of Differentiation

## A Mathematical Bridge to Artificial Intelligence

Differentiation supplies the mathematical language of change. An AI model is trained by changing parameters so that a loss becomes smaller. To understand that process carefully, students first need functions, limits, continuity, average and instantaneous rates of change, derivative rules, the chain rule, and one-variable optimization.

This notebook develops each idea from its background, carries calculations through every intermediate step, and then connects the mathematics to loss functions, backpropagation, and gradient descent.

**Recommended pacing:** three class meetings. The epsilon-delta argument and product-rule proof may be treated as instructor-led extensions.


> **How to use this notebook in class**
>
> Ask students to predict graphs and derivative signs before running code. Keep the distinction between a variable, a parameter, a model output, and a loss visible throughout the module. Open each instructor solution only after students have attempted the worksheet.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Functions, limits, and continuity | Models and loss functions must be evaluated near current parameter values |
| 2 | Derivative from a limit | Sensitivity of loss to a parameter |
| 3 | Derivative rules | Efficient differentiation of model formulas |
| 4 | Chain rule | Backpropagation through composed operations |
| 5 | Local extrema and derivative tests | Locating loss minima |
| 6 | Gradient descent in one variable | Iterative parameter training |
| Integrated lab | Fit a one-parameter model | Connect formula, derivative, update, and convergence |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable instructor-solution helper.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def tangent_line(function, derivative, point, x_values):
    return function(point) + derivative(point) * (x_values - point)


## Module learning objectives

By the end of the module, students should be able to:

1. distinguish variables, parameters, model outputs, and loss values;
2. estimate and compute limits and test continuity;
3. derive a derivative from the difference quotient;
4. apply power, sum, product, quotient, exponential, and logarithmic rules;
5. differentiate a composition using the chain rule;
6. trace local derivatives backward through a small computation graph;
7. find and classify critical points; and
8. carry out and interpret gradient-descent iterations.

## Prerequisite checkpoint

Students should be comfortable with algebraic simplification, function notation, graph coordinates, powers, exponentials, and logarithms. No prior calculus is assumed.


---

# Section 1: Functions, Limits, and Continuity

## 1. Why begin with functions?

A **function** is a rule that assigns exactly one output to every allowed input. We write

$$
y=f(x).
$$

The symbol $$x$$ is an **independent variable**, and $$y$$ is a **dependent variable** because its value depends on $$x.$$ The **domain** is the collection of allowed inputs, and the **range** is the collection of outputs actually produced.

In AI, a model is a function. A simple one-input model is

$$
\widehat{y}=f(x;w,b)=wx+b.
$$

Here $$x$$ is an input, while $$w$$ and $$b$$ are **parameters**: adjustable numbers learned during training. The symbol $$\widehat{y}$$ means a prediction, and the observed target is usually denoted by $$y.$$


## 2. From a model to a loss function

A **loss function** converts a prediction error into a numerical penalty. For one observation, squared error is

$$
L(w,b)=(\widehat{y}-y)^2.
$$

Substituting the model gives

$$
L(w,b)=(wx+b-y)^2.
$$

For fixed data $$x$$ and $$y,$$ this is a function of the adjustable parameters. Training asks: how should the parameters change so that $$L$$ decreases?

This question depends on what the loss does **near** the current parameter value. Limits formalize the word “near.”


In [ ]:
x_model = np.linspace(-2, 4, 300)
w_model, b_model = 1.5, 0.5
y_model = w_model*x_model + b_model

fig, ax = plt.subplots()
ax.plot(x_model, y_model, linewidth=2, label="model output")
ax.scatter([2], [w_model*2+b_model], s=70, color="crimson", zorder=3,
           label="one prediction")
ax.set_xlabel("input x")
ax.set_ylabel("prediction")
ax.set_title("A linear model is a function")
ax.legend()
plt.show()


## 3. The intuitive meaning of a limit

The statement

$$
\lim_{x\to a}f(x)=L
$$

means that values of $$f(x)$$ can be made as close as desired to $$L$$ by taking $$x$$ sufficiently close to $$a,$$ without requiring $$x=a.$$

The number $$a$$ is the **approach point**, and $$L$$ is the **limit value**. A limit describes nearby behavior, whereas $$f(a)$$ describes the value at the point.

### Detailed example: a removable hole

Consider

$$
f(x)=\frac{x^2-1}{x-1}.
$$

At $$x=1,$$ the displayed fraction is undefined because its denominator is zero. For $$x\ne 1,$$ factor the numerator:

$$
x^2-1=(x-1)(x+1).
$$

Therefore

$$
f(x)=\frac{(x-1)(x+1)}{x-1}=x+1,
\qquad x\ne 1.
$$

As $$x$$ approaches $$1,$$ the simplified expression approaches

$$
1+1=2.
$$

Hence

$$
\boxed{\lim_{x\to 1}\frac{x^2-1}{x-1}=2.}
$$

The limit exists even though the original function is not defined at $$x=1.$$


In [ ]:
x_limit_table = np.array([0.9, 0.99, 0.999, 1.001, 1.01, 1.1])
f_limit_table = (x_limit_table**2 - 1)/(x_limit_table - 1)
print("       x          f(x)")
for x_value, f_value in zip(x_limit_table, f_limit_table):
    print(f"{x_value:8.3f}    {f_value:10.6f}")

x_left = np.linspace(0, 0.995, 200)
x_right = np.linspace(1.005, 2, 200)
fig, ax = plt.subplots()
ax.plot(x_left, x_left+1, color="navy")
ax.plot(x_right, x_right+1, color="navy")
ax.scatter([1], [2], s=100, facecolors="white", edgecolors="crimson",
           linewidths=2, label="missing point")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Nearby values approach 2 even though f(1) is undefined")
ax.legend()
plt.show()


## 4. One-sided and two-sided limits

The **left-hand limit** uses inputs smaller than $$a$$:

$$
\lim_{x\to a^-}f(x).
$$

The **right-hand limit** uses inputs larger than $$a$$:

$$
\lim_{x\to a^+}f(x).
$$

A two-sided limit exists precisely when both one-sided limits exist and agree:

$$
\lim_{x\to a}f(x)=L
\quad\Longleftrightarrow\quad
\lim_{x\to a^-}f(x)=L
\text{ and }
\lim_{x\to a^+}f(x)=L.
$$

For the step function

$$
s(x)=
\begin{cases}
0,&x<0,\\
1,&x\geq 0,
\end{cases}
$$

the left-hand limit at zero is $$0,$$ while the right-hand limit is $$1.$$ Because they disagree, the two-sided limit does not exist.


## 5. Continuity

A function $$f$$ is **continuous at** $$x=a$$ when all three conditions hold:

1. $$f(a)$$ is defined;
2. $$\lim_{x\to a}f(x)$$ exists; and
3. $$\lim_{x\to a}f(x)=f(a).$$

Informally, a small change in the input produces a small change in the output near that point.

### Example: repairing a discontinuity

Define

$$
g(x)=
\begin{cases}
\dfrac{x^2-1}{x-1},&x\ne 1,\\
c,&x=1.
\end{cases}
$$

The earlier limit calculation gives

$$
\lim_{x\to 1}g(x)=2.
$$

Continuity requires

$$
g(1)=2.
$$

Therefore the missing value must be filled with

$$
\boxed{c=2.}
$$


## 6. A familiar AI activation: ReLU

The **rectified linear unit**, or **ReLU**, is

$$
\operatorname{ReLU}(z)=\max(0,z)
=
\begin{cases}
0,&z<0,\\
z,&z\geq 0.
\end{cases}
$$

At zero, both one-sided function limits equal zero and the function value is zero. ReLU is therefore continuous at zero. Later we will see that its left and right slopes disagree, so it is not differentiable there.


In [ ]:
z_relu = np.linspace(-3, 3, 400)
relu_values = np.maximum(0, z_relu)
fig, ax = plt.subplots()
ax.plot(z_relu, relu_values, linewidth=2)
ax.scatter([0], [0], color="crimson", zorder=3)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("pre-activation z")
ax.set_ylabel("ReLU(z)")
ax.set_title("ReLU is continuous but has a corner at zero")
plt.show()


## Optional mathematical precision: an epsilon-delta limit

The formal statement

$$
\lim_{x\to a}f(x)=L
$$

means: for every tolerance $$\varepsilon>0,$$ there is a distance $$\delta>0$$ such that

$$
0<|x-a|<\delta
\quad\Longrightarrow\quad
|f(x)-L|<\varepsilon.
$$

For example, prove

$$
\lim_{x\to 3}(2x+1)=7.
$$

We want

$$
|(2x+1)-7|<\varepsilon.
$$

Simplify:

$$
|(2x+1)-7|=|2x-6|=2|x-3|.
$$

Choose

$$
\delta=\frac{\varepsilon}{2}.
$$

Then

$$
0<|x-3|<\delta
$$

implies

$$
|(2x+1)-7|=2|x-3|<2\delta=\varepsilon.
$$

This proves the limit.


> **Common mistakes in Section 1**
>
> - Treating a parameter and an input variable as the same role.
> - Assuming a limit must equal the function value.
> - Substituting into an indeterminate form and stopping before simplifying.
> - Checking only one side of a two-sided limit.
> - Calling every continuous function differentiable.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Function | Rule assigning exactly one output to each allowed input |
| Domain | Set of allowed inputs |
| Range | Set of outputs actually produced |
| Parameter | Adjustable number controlling a model |
| Prediction | Model output, commonly written with a hat |
| Loss function | Numerical penalty measuring prediction error |
| Limit | Value approached by nearby function outputs |
| One-sided limit | Limit approached from only the left or right |
| Continuity | Agreement of function value and two-sided limit |
| ReLU | Piecewise-linear activation equal to the positive part of its input |


## Section 1 guided worksheet

1. For the model

   $$
   \widehat{y}=wx+b,
   $$

   identify the input, parameters, and output.

2. Compute

   $$
   \lim_{x\to 2}\frac{x^2-4}{x-2}.
   $$

3. Choose $$c$$ so that

   $$
   h(x)=
   \begin{cases}
   \dfrac{x^2-4}{x-2},&x\ne 2,\\
   c,&x=2
   \end{cases}
   $$

   is continuous at $$x=2.$$

4. Explain why a function with unequal one-sided limits cannot be continuous at that point.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. The input is $$x.$$ The adjustable parameters are $$w$$ and $$b.$$ The model output or prediction is $$\widehat{y}.$$

2. Factor the numerator:

   $$
   x^2-4=(x-2)(x+2).
   $$

   For $$x\ne 2,$$

   $$
   \frac{x^2-4}{x-2}=x+2.
   $$

   Therefore

   $$
   \lim_{x\to 2}\frac{x^2-4}{x-2}
   =\lim_{x\to 2}(x+2)=4.
   $$

3. Continuity requires the function value to equal the limit. Thus

   $$
   \boxed{c=4.}
   $$

4. A two-sided limit exists only when the left-hand and right-hand limits agree. If they disagree, condition 2 in the definition of continuity fails.
""")


---

# Section 2: The Derivative as an Instantaneous Rate of Change

## 1. Average change before instantaneous change

For a function $$f,$$ the **average rate of change** from $$x=a$$ to $$x=a+h$$ is

$$
\frac{f(a+h)-f(a)}{(a+h)-a}
=\frac{f(a+h)-f(a)}{h}.
$$

Geometrically, this is the slope of a **secant line** through two points on the graph. To obtain the slope at one point, move the second point toward the first by letting $$h\to 0.$$


## 2. Definition of the derivative

The **derivative of** $$f$$ **at** $$a$$ is

$$
f'(a)=\lim_{h\to 0}\frac{f(a+h)-f(a)}{h},
$$

provided this limit exists.

The expression

$$
\frac{f(a+h)-f(a)}{h}
$$

is the **difference quotient**. The derivative is simultaneously:

- the slope of the tangent line;
- an instantaneous rate of change; and
- the local sensitivity of the output to the input.

If $$f'(a)>0,$$ the function is locally increasing. If $$f'(a)<0,$$ it is locally decreasing. A large magnitude means strong local sensitivity.


## 3. Derive the derivative of a square from the definition

Let

$$
f(x)=x^2.
$$

Begin with the definition at an arbitrary point $$x$$:

$$
f'(x)=\lim_{h\to 0}\frac{f(x+h)-f(x)}{h}.
$$

Substitute the function:

$$
f'(x)=\lim_{h\to 0}\frac{(x+h)^2-x^2}{h}.
$$

Expand the square:

$$
(x+h)^2=x^2+2xh+h^2.
$$

Therefore

$$
f'(x)=\lim_{h\to 0}\frac{x^2+2xh+h^2-x^2}{h}.
$$

Cancel $$x^2$$ and factor $$h$$:

$$
f'(x)=\lim_{h\to 0}\frac{2xh+h^2}{h}
=\lim_{h\to 0}\frac{h(2x+h)}{h}.
$$

For $$h\ne 0,$$ cancel $$h$$:

$$
f'(x)=\lim_{h\to 0}(2x+h).
$$

Now take the limit:

$$
\boxed{f'(x)=2x.}
$$

At $$x=2,$$ the tangent slope is

$$
f'(2)=2(2)=4.
$$


In [ ]:
f_square = lambda x: x**2
df_square = lambda x: 2*x
a_tangent = 2.0
x_tangent = np.linspace(-0.25, 4.25, 400)
y_tangent = f_square(x_tangent)
tangent_values = tangent_line(f_square, df_square, a_tangent, x_tangent)

fig, ax = plt.subplots()
ax.plot(x_tangent, y_tangent, linewidth=2, label="f(x) = x squared")
ax.plot(x_tangent, tangent_values, linestyle="--", linewidth=2,
        label="tangent at x = 2")
ax.scatter([a_tangent], [f_square(a_tangent)], color="crimson", zorder=3)
ax.set_ylim(-1, 15)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("The derivative is the tangent-line slope")
ax.legend()
plt.show()


## 4. Derivative notation and units

Equivalent notations include

$$
f'(x),
\qquad
\frac{df}{dx},
\qquad
\frac{dy}{dx}.
$$

If distance is measured in kilometers and time in hours, then its derivative has units kilometers per hour. If loss is dimensionless and a parameter is measured in some parameter unit, then

$$
\frac{dL}{dw}
$$

means loss change per unit change in $$w.$$

The derivative is a local linear approximation:

$$
f(a+\Delta x)\approx f(a)+f'(a)\Delta x
$$

for sufficiently small $$\Delta x.$$


## 5. Differentiability implies continuity

If $$f'(a)$$ exists, then $$f$$ must be continuous at $$a.$$ To see why, write

$$
f(a+h)-f(a)
=h\left(\frac{f(a+h)-f(a)}{h}\right).
$$

As $$h\to 0,$$ the first factor approaches zero and the second approaches the finite number $$f'(a).$$ Therefore

$$
\lim_{h\to 0}\bigl(f(a+h)-f(a)\bigr)=0,
$$

so

$$
\lim_{h\to 0}f(a+h)=f(a).
$$

That is continuity at $$a.$$

The converse is false. ReLU is continuous at zero, but its left slope is zero and its right slope is one. Its derivative at zero does not exist in the ordinary two-sided sense.


## 6. Numerical derivatives and step-size error

A computer can approximate a derivative with the centered difference

$$
f'(x)\approx \frac{f(x+h)-f(x-h)}{2h}.
$$

Moderately small $$h$$ improves the approximation. An extremely small $$h$$ can cause floating-point cancellation because two nearly equal numbers are subtracted.


In [ ]:
def centered_difference(function, point, step):
    return (function(point + step) - function(point - step))/(2*step)

steps_numeric = np.logspace(-16, -1, 100)
approximations_numeric = np.array([
    centered_difference(np.sin, 1.0, h) for h in steps_numeric
])
errors_numeric = np.abs(approximations_numeric - np.cos(1.0))

fig, ax = plt.subplots()
ax.loglog(steps_numeric, errors_numeric)
ax.set_xlabel("step size h")
ax.set_ylabel("absolute error")
ax.set_title("A numerical derivative has truncation and roundoff error")
plt.show()


## 7. Direct AI example: sensitivity of squared loss

Use the one-parameter model

$$
\widehat{y}=wx
$$

with one observation

$$
x=2,
\qquad
y=5.
$$

The loss is

$$
L(w)=(wx-y)^2=(2w-5)^2.
$$

At $$w=1,$$

$$
L(1)=(2-5)^2=9.
$$

After we learn the chain rule, we will obtain

$$
L'(w)=2(2w-5)(2)=4(2w-5).
$$

Thus

$$
L'(1)=4(2-5)=-12.
$$

The negative sign says that a small increase in $$w$$ should decrease the loss. For example, increasing $$w$$ from $$1$$ to $$1.1$$ changes the loss to

$$
L(1.1)=(2.2-5)^2=7.84,
$$

which is smaller than $$9.$$


> **Common mistakes in Section 2**
>
> - Confusing the difference quotient with the derivative before taking a limit.
> - Substituting $$h=0$$ before simplifying.
> - Treating the derivative as a global slope rather than a local one.
> - Ignoring units or the sign of a derivative.
> - Assuming a continuous corner must have a derivative.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Average rate of change | Output change divided by input change over an interval |
| Secant line | Line through two points of a graph |
| Difference quotient | Algebraic average-rate expression used before the limit |
| Derivative | Limit of the difference quotient |
| Tangent line | Local line whose slope is the derivative |
| Sensitivity | Local output response to a small input change |
| Differentiable | Having a finite two-sided derivative at the point |
| Centered difference | Symmetric numerical derivative approximation |


## Section 2 guided worksheet

1. Use the derivative definition to find the derivative of

   $$
   f(x)=3x+2.
   $$

2. For

   $$
   f(x)=x^2,
   $$

   compute the secant slope from $$x=2$$ to $$x=2.1$$ and compare it with the exact tangent slope at $$x=2.$$

3. Explain why ReLU is continuous but not differentiable at zero.

4. If $$L'(w)>0,$$ which small direction of change is expected to reduce the loss?


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
1. Start from the definition:

   $$
   f'(x)=\lim_{h\to 0}\frac{3(x+h)+2-(3x+2)}{h}.
   $$

   Simplify the numerator:

   $$
   3x+3h+2-3x-2=3h.
   $$

   Therefore

   $$
   f'(x)=\lim_{h\to 0}\frac{3h}{h}
   =\lim_{h\to 0}3
   =\boxed{3}.
   $$

2. The secant slope is

   $$
   \frac{f(2.1)-f(2)}{2.1-2}
   =\frac{(2.1)^2-2^2}{0.1}
   =\frac{4.41-4}{0.1}
   =4.1.
   $$

   The exact tangent slope is

   $$
   f'(2)=2(2)=4.
   $$

   The nearby secant slope $$4.1$$ approximates the tangent slope $$4.$$

3. Both one-sided function limits equal zero, so ReLU is continuous. Its left slope is zero and its right slope is one, so the two-sided derivative does not exist.

4. A positive derivative means increasing $$w$$ locally increases the loss. A small negative change in $$w$$ is expected to reduce it.
""")


---

# Section 3: Fundamental Differentiation Rules

## 1. Core rules

For differentiable functions $$f$$ and $$g$$ and a constant $$c,$$ the main rules are:

| Rule | Formula |
|---|---|
| Constant | $$\dfrac{d}{dx}c=0$$ |
| Power | $$\dfrac{d}{dx}x^n=nx^{n-1}$$ |
| Constant multiple | $$\dfrac{d}{dx}\bigl(cf(x)\bigr)=cf'(x)$$ |
| Sum | $$\dfrac{d}{dx}\bigl(f(x)+g(x)\bigr)=f'(x)+g'(x)$$ |
| Product | $$\dfrac{d}{dx}\bigl(f(x)g(x)\bigr)=f'(x)g(x)+f(x)g'(x)$$ |
| Quotient | $$\dfrac{d}{dx}\left(\dfrac{f(x)}{g(x)}\right)=\dfrac{f'(x)g(x)-f(x)g'(x)}{g(x)^2}$$ |

The quotient rule requires $$g(x)\ne 0.$$


## 2. Detailed polynomial example

Differentiate

$$
p(x)=4x^5-3x^2+7x-9.
$$

Apply the sum rule term by term:

$$
p'(x)
=\frac{d}{dx}(4x^5)
-\frac{d}{dx}(3x^2)
+\frac{d}{dx}(7x)
-\frac{d}{dx}(9).
$$

Apply the constant-multiple and power rules:

$$
\frac{d}{dx}(4x^5)=4(5x^4)=20x^4,
$$

$$
\frac{d}{dx}(3x^2)=3(2x)=6x,
$$

$$
\frac{d}{dx}(7x)=7,
$$

and

$$
\frac{d}{dx}(9)=0.
$$

Therefore

$$
\boxed{p'(x)=20x^4-6x+7.}
$$


## 3. Why the product rule has two terms

Starting from the difference quotient,

$$
\frac{f(x+h)g(x+h)-f(x)g(x)}{h},
$$

add and subtract $$f(x+h)g(x)$$ in the numerator:

$$
\frac{f(x+h)g(x+h)-f(x+h)g(x)+f(x+h)g(x)-f(x)g(x)}{h}.
$$

Group the terms:

$$
f(x+h)\frac{g(x+h)-g(x)}{h}
+g(x)\frac{f(x+h)-f(x)}{h}.
$$

As $$h\to 0,$$ differentiability gives continuity, so $$f(x+h)\to f(x).$$ The two difference quotients approach the corresponding derivatives. Hence

$$
\boxed{\frac{d}{dx}\bigl(fg\bigr)=fg'+gf'.}
$$

Both terms are necessary because both factors change.


## 4. Product and quotient examples

Let

$$
q(x)=(x^2+1)(3x-4).
$$

Set

$$
f(x)=x^2+1,
\qquad
g(x)=3x-4.
$$

Then

$$
f'(x)=2x,
\qquad
g'(x)=3.
$$

The product rule gives

$$
q'(x)=2x(3x-4)+(x^2+1)(3).
$$

Expand:

$$
q'(x)=6x^2-8x+3x^2+3
=\boxed{9x^2-8x+3}.
$$

For

$$
r(x)=\frac{x^2+1}{x},
\qquad x\ne 0,
$$

the quotient rule gives

$$
r'(x)=\frac{(2x)(x)-(x^2+1)(1)}{x^2}
=\frac{2x^2-x^2-1}{x^2}
=\boxed{\frac{x^2-1}{x^2}}.
$$


## 5. Exponential and logarithmic functions

The natural exponential function is the unique exponential whose derivative equals itself:

$$
\frac{d}{dx}e^x=e^x.
$$

Its inverse is the natural logarithm:

$$
\frac{d}{dx}\ln x=\frac{1}{x},
\qquad x>0.
$$

These functions occur throughout AI. Exponentials convert arbitrary real scores into positive quantities, as in sigmoid and softmax. Logarithms turn products into sums and appear in cross-entropy and likelihood losses.

For example, if

$$
f(x)=x^2e^x,
$$

then the product rule gives

$$
f'(x)=(2x)e^x+x^2e^x
=e^x(x^2+2x).
$$


## 6. Mean squared error as a differentiable function

For predictions

$$
\widehat{y}_i=wx_i
$$

and observations $$y_i,$$ the **mean squared error** is

$$
L(w)=\frac{1}{n}\sum_{i=1}^{n}(wx_i-y_i)^2.
$$

Expanding one squared term gives

$$
(wx_i-y_i)^2=w^2x_i^2-2wx_i y_i+y_i^2.
$$

Differentiate term by term:

$$
\frac{d}{dw}(wx_i-y_i)^2
=2wx_i^2-2x_i y_i
=2x_i(wx_i-y_i).
$$

Therefore

$$
\boxed{L'(w)=\frac{2}{n}\sum_{i=1}^{n}x_i(wx_i-y_i).}
$$

The residual $$wx_i-y_i$$ is multiplied by the input $$x_i,$$ so observations with larger input magnitude can contribute more strongly to the derivative.


In [ ]:
x_mse = np.array([1.0, 2.0, 3.0])
y_mse = np.array([2.0, 4.0, 6.0])

def mse_one_parameter(w):
    return np.mean((w*x_mse - y_mse)**2)

def dmse_one_parameter(w):
    return (2/len(x_mse))*np.sum(x_mse*(w*x_mse-y_mse))

print("MSE at w = 1:", mse_one_parameter(1.0))
print("Derivative at w = 1:", dmse_one_parameter(1.0))


> **Common mistakes in Section 3**
>
> - Reducing the exponent without multiplying by the old exponent.
> - Using $$f'g'$$ for a product instead of $$f'g+fg'.$$
> - Reversing the subtraction order in the quotient rule.
> - Forgetting that the denominator is squared in the quotient rule.
> - Treating $$e^x$$ like a power $$x^n.$$
> - Omitting the averaging factor in mean squared error.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Power rule | Rule for differentiating a power of the variable |
| Product rule | Rule accounting for changes in both multiplied factors |
| Quotient rule | Rule for differentiating a ratio |
| Natural exponential | Function $$e^x$$ whose derivative equals itself |
| Natural logarithm | Inverse of the natural exponential |
| Residual | Prediction minus observed target |
| Mean squared error | Average of squared residuals |


## Section 3 guided worksheet

Differentiate each function and show the rule used:

1. $$f(x)=5x^4-2x+8.$$
2. $$g(x)=(x^3+1)(x^2-2).$$
3. $$h(x)=\dfrac{e^x}{x}.$$ 
4. For two observations, write the derivative of

   $$
   L(w)=\frac{1}{2}\left[(wx_1-y_1)^2+(wx_2-y_2)^2\right].
   $$


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
1. Apply the power and sum rules:

   $$
   f'(x)=5(4x^3)-2+0=\boxed{20x^3-2}.
   $$

2. Apply the product rule:

   $$
   g'(x)=3x^2(x^2-2)+(x^3+1)(2x).
   $$

   Expanding gives

   $$
   g'(x)=3x^4-6x^2+2x^4+2x
   =\boxed{5x^4-6x^2+2x}.
   $$

3. Apply the quotient rule:

   $$
   h'(x)=\frac{e^x(x)-e^x(1)}{x^2}
   =\boxed{\frac{e^x(x-1)}{x^2}},
   \qquad x\ne 0.
   $$

4. Differentiate the two squared residuals:

   $$
   L'(w)=\frac{1}{2}\left[2x_1(wx_1-y_1)+2x_2(wx_2-y_2)\right].
   $$

   Cancel the factor of two:

   $$
   \boxed{L'(w)=x_1(wx_1-y_1)+x_2(wx_2-y_2).}
   $$
""")


---

# Section 4: The Chain Rule and Backpropagation

## 1. Composition of functions

A **composition** feeds the output of one function into another. If

$$
u=g(x)
$$

and

$$
y=f(u),
$$

then

$$
y=f(g(x)).
$$

The chain rule states

$$
\boxed{\frac{dy}{dx}=\frac{dy}{du}\frac{du}{dx}.}
$$

Equivalently,

$$
\frac{d}{dx}f(g(x))=f'(g(x))g'(x).
$$

The derivative of the outer function is evaluated at the inner output and multiplied by the derivative of the inner function.


## 2. Detailed chain-rule example

Differentiate

$$
y=(3x^2+1)^4.
$$

Identify an inner quantity:

$$
u=3x^2+1.
$$

Then

$$
y=u^4.
$$

Differentiate the outer function with respect to $$u$$:

$$
\frac{dy}{du}=4u^3.
$$

Differentiate the inner function with respect to $$x$$:

$$
\frac{du}{dx}=6x.
$$

Multiply the local derivatives:

$$
\frac{dy}{dx}
=\frac{dy}{du}\frac{du}{dx}
=4u^3(6x)
=24xu^3.
$$

Replace $$u$$ with $$3x^2+1$$:

$$
\boxed{\frac{dy}{dx}=24x(3x^2+1)^3.}
$$


## 3. Why local derivatives multiply

For small changes, the inner function approximately satisfies

$$
\Delta u\approx \frac{du}{dx}\Delta x.
$$

The outer function approximately satisfies

$$
\Delta y\approx \frac{dy}{du}\Delta u.
$$

Substituting the first approximation into the second gives

$$
\Delta y
\approx \frac{dy}{du}\left(\frac{du}{dx}\Delta x\right)
=\left(\frac{dy}{du}\frac{du}{dx}\right)\Delta x.
$$

Thus the overall sensitivity is the product

$$
\frac{dy}{dx}=\frac{dy}{du}\frac{du}{dx}.
$$

This multiplication of local sensitivities is the central arithmetic of backpropagation.


## 4. Computation graphs and backpropagation

A **computation graph** breaks a formula into simple operations. **Forward propagation** evaluates operations from inputs to loss. **Backpropagation** traverses the graph in reverse, multiplying local derivatives to compute how the final loss depends on earlier quantities.

Consider one sigmoid neuron:

$$
z=wx+b,
$$

$$
\widehat{y}=\sigma(z)=\frac{1}{1+e^{-z}},
$$

$$
L=\frac{1}{2}(\widehat{y}-y)^2.
$$

The chain from $$w$$ to $$L$$ is

$$
w\longrightarrow z\longrightarrow \widehat{y}\longrightarrow L.
$$


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis("off")
nodes = [(0.08, "parameter w"), (0.34, "z = wx + b"),
         (0.62, "prediction"), (0.90, "loss L")]
for position, label in nodes:
    ax.text(position, 0.5, label, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#dbeafe", edgecolor="#1d4ed8"))
for start, end in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", xy=(end[0]-0.08, 0.5), xytext=(start[0]+0.08, 0.5),
                arrowprops=dict(arrowstyle="->", linewidth=2))
ax.set_title("Forward evaluation moves right; derivatives are propagated left")
plt.show()


## 5. Fully worked one-neuron derivative

Use

$$
x=2,
\qquad
w=0.5,
\qquad
b=-0.5,
\qquad
y=1.
$$

### Forward pass

First,

$$
z=wx+b=(0.5)(2)-0.5=0.5.
$$

Then

$$
\widehat{y}=\sigma(0.5)
=\frac{1}{1+e^{-0.5}}
\approx 0.622459.
$$

Finally,

$$
L=\frac{1}{2}(0.622459-1)^2
\approx 0.071268.
$$

### Backward pass

The loss derivative is

$$
\frac{dL}{d\widehat{y}}=\widehat{y}-y
\approx -0.377541.
$$

The sigmoid derivative is

$$
\frac{d\widehat{y}}{dz}
=\sigma(z)(1-\sigma(z))
\approx (0.622459)(0.377541)
\approx 0.235004.
$$

The linear derivative is

$$
\frac{dz}{dw}=x=2.
$$

Multiply along the path:

$$
\frac{dL}{dw}
=\frac{dL}{d\widehat{y}}
\frac{d\widehat{y}}{dz}
\frac{dz}{dw}.
$$

Therefore

$$
\frac{dL}{dw}
\approx (-0.377541)(0.235004)(2)
\approx \boxed{-0.177447}.
$$

The negative derivative says that a small increase in $$w$$ should reduce this observation's loss.


In [ ]:
x_neuron, w_neuron, b_neuron, y_neuron = 2.0, 0.5, -0.5, 1.0
z_neuron = w_neuron*x_neuron + b_neuron
yhat_neuron = 1/(1+np.exp(-z_neuron))
loss_neuron = 0.5*(yhat_neuron-y_neuron)**2
dL_dyhat = yhat_neuron-y_neuron
dyhat_dz = yhat_neuron*(1-yhat_neuron)
dz_dw = x_neuron
dL_dw_neuron = dL_dyhat*dyhat_dz*dz_dw

print("z =", z_neuron)
print("prediction =", yhat_neuron)
print("loss =", loss_neuron)
print("dL/dw =", dL_dw_neuron)


> **Common mistakes in Section 4**
>
> - Differentiating only the outer function.
> - Adding local derivatives instead of multiplying along a path.
> - Forgetting to evaluate the outer derivative at the inner value.
> - Mixing forward values with backward derivatives.
> - Reversing the direction of backpropagation.
> - Treating backpropagation as a different derivative rule rather than organized chain-rule computation.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Composition | Function formed by feeding one function's output into another |
| Chain rule | Rule multiplying derivatives through a composition |
| Computation graph | Network of simple operations representing a calculation |
| Forward propagation | Evaluation from inputs and parameters to model output and loss |
| Backpropagation | Reverse accumulation of derivatives through the graph |
| Local derivative | Derivative of one operation with respect to its immediate input |
| Sigmoid | Smooth activation mapping real numbers into the interval from zero to one |


## Section 4 guided worksheet

1. Differentiate

   $$
   f(x)=(2x-1)^5.
   $$

2. Differentiate

   $$
   g(x)=e^{x^2+3x}.
   $$

3. For

   $$
   z=wx+b,
   \qquad
   L=z^2,
   $$

   compute $$dL/dw$$ by listing and multiplying the local derivatives.

4. Explain in words why backpropagation moves in the opposite direction from the forward pass.


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. Let $$u=2x-1.$$ Then

   $$
   \frac{df}{du}=5u^4,
   \qquad
   \frac{du}{dx}=2.
   $$

   Thus

   $$
   f'(x)=5(2x-1)^4(2)=\boxed{10(2x-1)^4}.
   $$

2. Let $$u=x^2+3x.$$ Then

   $$
   \frac{dg}{du}=e^u,
   \qquad
   \frac{du}{dx}=2x+3.
   $$

   Therefore

   $$
   g'(x)=\boxed{e^{x^2+3x}(2x+3)}.
   $$

3. The local derivatives are

   $$
   \frac{dL}{dz}=2z
   $$

   and

   $$
   \frac{dz}{dw}=x.
   $$

   Multiply them:

   $$
   \frac{dL}{dw}=\frac{dL}{dz}\frac{dz}{dw}
   =2zx
   =\boxed{2x(wx+b)}.
   $$

4. The forward pass first creates later quantities from earlier ones. To learn how the final loss depends on an earlier parameter, the chain rule starts at the loss and follows dependency paths backward, multiplying local sensitivities.
""")


---

# Section 5: Optimization and Local Extrema

## 1. What optimization asks

**Optimization** seeks inputs that make an objective function as small or as large as possible. AI training is usually a minimization problem:

$$
\min_w L(w).
$$

A point $$w^*$$ is a **local minimum** if its loss is no larger than the loss at all sufficiently nearby points. It is a **global minimum** if its loss is no larger than the loss at every point in the domain.

A **critical point** is an interior-domain point where

$$
L'(w)=0
$$

or where the derivative does not exist. Critical points are candidates, not automatic minima.


## 2. Why differentiable interior extrema are critical

Suppose a differentiable function has a local minimum at an interior point $$a.$$ For small positive $$h,$$

$$
f(a+h)-f(a)\geq 0,
$$

so

$$
\frac{f(a+h)-f(a)}{h}\geq 0.
$$

For small negative $$h,$$ the numerator remains nonnegative while the denominator is negative, so

$$
\frac{f(a+h)-f(a)}{h}\leq 0.
$$

If the two-sided derivative exists, the right and left limits must agree. The only number that is both nonnegative and nonpositive is zero. Therefore

$$
f'(a)=0.
$$

This result is often called Fermat's theorem for stationary points.


## 3. First- and second-derivative tests

At a critical point $$c$$:

- if $$f'$$ changes from negative to positive, $$c$$ is a local minimum;
- if $$f'$$ changes from positive to negative, $$c$$ is a local maximum;
- if the sign does not change, the point may be neither.

When the second derivative exists:

$$
f''(c)>0
\quad\Longrightarrow\quad
\text{local minimum},
$$

$$
f''(c)<0
\quad\Longrightarrow\quad
\text{local maximum}.
$$

If $$f''(c)=0,$$ the test is inconclusive.

The second derivative measures **curvature**. Positive curvature means the graph bends upward locally; negative curvature means it bends downward.


## 4. Detailed nonconvex example

Let

$$
f(x)=x^4-4x^2+3.
$$

Differentiate:

$$
f'(x)=4x^3-8x.
$$

Factor:

$$
f'(x)=4x(x^2-2).
$$

Set the derivative equal to zero:

$$
4x(x^2-2)=0.
$$

Therefore the critical points are

$$
x=0,
\qquad
x=\sqrt{2},
\qquad
x=-\sqrt{2}.
$$

The second derivative is

$$
f''(x)=12x^2-8.
$$

At $$x=0,$$

$$
f''(0)=-8<0,
$$

so $$x=0$$ is a local maximum.

At $$x=\pm\sqrt{2},$$

$$
f''(\pm\sqrt{2})=12(2)-8=16>0,
$$

so both are local minima. Their values are

$$
f(\pm\sqrt{2})=(\sqrt{2})^4-4(\sqrt{2})^2+3
=4-8+3=-1.
$$


In [ ]:
x_opt = np.linspace(-2.5, 2.5, 500)
f_opt = x_opt**4 - 4*x_opt**2 + 3
critical_opt = np.array([-np.sqrt(2), 0, np.sqrt(2)])
critical_values_opt = critical_opt**4 - 4*critical_opt**2 + 3

fig, ax = plt.subplots()
ax.plot(x_opt, f_opt, linewidth=2)
ax.scatter(critical_opt, critical_values_opt, color=["green", "crimson", "green"],
           s=70, zorder=3)
ax.set_ylim(-2, 8)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Critical points can be minima or maxima")
plt.show()


> **Common mistakes in Section 5**
>
> - Declaring every point with zero derivative a minimum.
> - Forgetting points where the derivative is undefined.
> - Using the second-derivative test when the second derivative equals zero.
> - Confusing local and global optimality.
> - Ignoring endpoints when optimizing on a closed interval.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Objective function | Function to be minimized or maximized |
| Local minimum | Point no worse than all sufficiently nearby points |
| Global minimum | Point no worse than every point in the domain |
| Critical point | Candidate point where the derivative is zero or undefined |
| Stationary point | Point where the derivative equals zero |
| Curvature | Local bending measured by the second derivative |
| First-derivative test | Classification using changes in derivative sign |
| Second-derivative test | Classification using curvature at a critical point |


## Section 5 guided worksheet

For

$$
f(x)=x^3-3x^2-9x+5,
$$

1. compute $$f'(x)$$;
2. find all critical points;
3. compute $$f''(x)$$; and
4. classify each critical point using the second-derivative test.

Then explain why the equation $$f'(x)=0$$ alone does not complete an optimization problem.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
Differentiate:

$$
f'(x)=3x^2-6x-9.
$$

Factor:

$$
f'(x)=3(x^2-2x-3)=3(x-3)(x+1).
$$

Thus

$$
f'(x)=0
$$

at

$$
x=3
\qquad\text{and}\qquad
x=-1.
$$

The second derivative is

$$
f''(x)=6x-6.
$$

At $$x=3,$$

$$
f''(3)=18-6=12>0,
$$

so $$x=3$$ is a local minimum.

At $$x=-1,$$

$$
f''(-1)=-6-6=-12<0,
$$

so $$x=-1$$ is a local maximum.

Solving $$f'(x)=0$$ only produces candidates. Classification, domain endpoints, nondifferentiable points, and global comparisons may still be required.
""")


---

# Section 6: Gradient Descent in One Variable

## 1. The update rule

**Gradient descent** repeatedly moves a parameter opposite the derivative:

$$
w_{k+1}=w_k-\eta L'(w_k).
$$

Here:

- $$k$$ is the iteration number;
- $$w_k$$ is the current parameter;
- $$L'(w_k)$$ is the current slope; and
- $$\eta>0$$ is the **learning rate**, or step-size multiplier.

The derivative supplies a direction, while the learning rate controls how far the algorithm moves.


## 2. Fully worked iterations

Let

$$
L(w)=(w-3)^2.
$$

Then

$$
L'(w)=2(w-3).
$$

Choose

$$
w_0=0,
\qquad
\eta=0.1.
$$

The first update is

$$
w_1=w_0-0.1L'(w_0).
$$

Because

$$
L'(0)=2(0-3)=-6,
$$

we obtain

$$
w_1=0-0.1(-6)=0.6.
$$

Next,

$$
L'(0.6)=2(0.6-3)=-4.8,
$$

so

$$
w_2=0.6-0.1(-4.8)=1.08.
$$

Next,

$$
L'(1.08)=2(1.08-3)=-3.84,
$$

so

$$
w_3=1.08-0.1(-3.84)=1.464.
$$

The sequence moves toward the minimizer $$w=3.$$


## 3. Why the learning rate matters

For this quadratic loss,

$$
w_{k+1}=w_k-2\eta(w_k-3).
$$

Subtract $$3$$ from both sides:

$$
w_{k+1}-3=(1-2\eta)(w_k-3).
$$

The distance from the optimum is multiplied by $$1-2\eta$$ at every step.

- If $$\eta=0.1,$$ the multiplier is $$0.8,$$ so the error steadily shrinks.
- If $$\eta=0.5,$$ the multiplier is $$0,$$ so the exact minimum is reached in one step.
- If $$\eta=0.8,$$ the multiplier is $$-0.6,$$ so iterates alternate across the minimum while shrinking.
- If $$\eta=1.1,$$ the multiplier is $$-1.2,$$ so the error grows and the method diverges.


In [ ]:
def gradient_descent_quadratic(learning_rate, iterations=12, initial=0.0):
    values = [initial]
    for _ in range(iterations):
        current = values[-1]
        values.append(current - learning_rate*2*(current-3))
    return np.array(values)

rates_gd = [0.1, 0.5, 0.8, 1.1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for rate in rates_gd:
    path = gradient_descent_quadratic(rate)
    axes[0].plot(path, marker="o", label=f"learning rate = {rate}")
    axes[1].semilogy(np.maximum(np.abs(path-3), 1e-14), marker="o",
                    label=f"learning rate = {rate}")
axes[0].axhline(3, color="black", linestyle="--", label="optimum")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("parameter w")
axes[0].set_title("Parameter paths")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("distance from optimum")
axes[1].set_title("Convergence or divergence")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## 4. Training vocabulary

In practical AI training:

- an **iteration** is one parameter update;
- a **batch** is the subset of data used for one update;
- a **minibatch** is a batch smaller than the full dataset;
- an **epoch** is one pass through the training dataset;
- the **training loss** is evaluated on fitting data; and
- the **validation loss** is evaluated on held-out data to monitor generalization.

The derivative of a minibatch loss is an estimate of the full-data derivative. This makes large-scale learning computationally feasible but introduces variation between updates.


> **Common mistakes in Section 6**
>
> - Moving in the same direction as a positive derivative when minimizing.
> - Treating the learning rate as though it were a derivative.
> - Assuming a larger step always trains faster.
> - Calling one iteration an epoch regardless of dataset coverage.
> - Judging generalization only from training loss.
> - Expecting every minibatch to produce the same derivative.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Gradient descent | Iterative movement opposite the derivative or gradient |
| Learning rate | Positive multiplier controlling update size |
| Iteration | One parameter-update step |
| Batch | Data subset used to compute an update |
| Minibatch | Batch smaller than the full training dataset |
| Epoch | One pass through the training dataset |
| Training loss | Loss evaluated on data used for fitting |
| Validation loss | Loss evaluated on held-out data |
| Convergence | Iterates or losses approaching a stable limit |


## Section 6 guided worksheet

Use

$$
L(w)=(w-4)^2,
\qquad
w_0=1,
\qquad
\eta=0.25.
$$

1. Compute $$L'(w).$$
2. Compute $$w_1,$$ $$w_2,$$ and $$w_3$$ with all intermediate derivatives.
3. Derive a recurrence for the error $$e_k=w_k-4.$$
4. Decide whether the error shrinks, oscillates, or grows.


In [ ]:
reveal_markdown("Open Section 6 solutions", r"""
1. The derivative is

   $$
   L'(w)=2(w-4).
   $$

2. At $$w_0=1,$$

   $$
   L'(1)=2(1-4)=-6,
   $$

   so

   $$
   w_1=1-0.25(-6)=2.5.
   $$

   Next,

   $$
   L'(2.5)=2(2.5-4)=-3,
   $$

   so

   $$
   w_2=2.5-0.25(-3)=3.25.
   $$

   Next,

   $$
   L'(3.25)=2(3.25-4)=-1.5,
   $$

   so

   $$
   w_3=3.25-0.25(-1.5)=3.625.
   $$

3. The update is

   $$
   w_{k+1}=w_k-0.25\bigl(2(w_k-4)\bigr).
   $$

   Subtracting $$4$$ gives

   $$
   e_{k+1}=(1-0.5)e_k=0.5e_k.
   $$

4. The error is multiplied by $$0.5$$ each time, so it shrinks without alternating signs.
""")


---

# Integrated Instructor Lab: Fit a One-Parameter Linear Model

Use the data

$$
(x_1,y_1)=(1,2),
\qquad
(x_2,y_2)=(2,4),
\qquad
(x_3,y_3)=(3,6)
$$

and the model

$$
\widehat{y}=wx.
$$

The mean squared error is

$$
L(w)=\frac{1}{3}\left[(w-2)^2+(2w-4)^2+(3w-6)^2\right].
$$

Because

$$
2w-4=2(w-2)
$$

and

$$
3w-6=3(w-2),
$$

we obtain

$$
L(w)=\frac{1}{3}\left[1^2(w-2)^2+2^2(w-2)^2+3^2(w-2)^2\right].
$$

Factor:

$$
L(w)=\frac{1+4+9}{3}(w-2)^2
=\frac{14}{3}(w-2)^2.
$$

Differentiate:

$$
L'(w)=\frac{28}{3}(w-2).
$$

The unique critical point is

$$
w=2.
$$

Since

$$
L''(w)=\frac{28}{3}>0,
$$

this point is the global minimum.


In [ ]:
w_grid_lab5 = np.linspace(-0.5, 3.5, 400)
loss_grid_lab5 = np.array([mse_one_parameter(w) for w in w_grid_lab5])

w_path_lab5 = [0.0]
eta_lab5 = 0.05
for _ in range(15):
    w_current = w_path_lab5[-1]
    w_path_lab5.append(w_current - eta_lab5*dmse_one_parameter(w_current))
w_path_lab5 = np.array(w_path_lab5)
loss_path_lab5 = np.array([mse_one_parameter(w) for w in w_path_lab5])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(w_grid_lab5, loss_grid_lab5, linewidth=2)
axes[0].scatter(w_path_lab5, loss_path_lab5, c=np.arange(len(w_path_lab5)),
                cmap="viridis", zorder=3)
axes[0].set_xlabel("parameter w")
axes[0].set_ylabel("mean squared error")
axes[0].set_title("Gradient descent follows the loss slope")
axes[1].plot(w_path_lab5, marker="o")
axes[1].axhline(2, color="black", linestyle="--", label="exact optimum")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("parameter w")
axes[1].set_title("Training trajectory")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Final parameter:", w_path_lab5[-1])
print("Final loss:", loss_path_lab5[-1])


## Integrated lab tasks

1. Verify the first three gradient-descent updates by hand using $$w_0=0$$ and $$\eta=0.05.$$
2. Explain why the derivative is negative when $$w<2$$ and positive when $$w>2.$$
3. Predict what will happen if the learning rate is made much larger.
4. Explain the roles of the data, model, loss, derivative, and update rule as five distinct objects.


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
The derivative is

$$
L'(w)=\frac{28}{3}(w-2).
$$

At $$w_0=0,$$

$$
L'(0)=\frac{28}{3}(-2)=-\frac{56}{3}.
$$

Thus

$$
w_1=0-0.05\left(-\frac{56}{3}\right)
=\frac{14}{15}
\approx 0.933333.
$$

At $$w_1=14/15,$$

$$
L'(w_1)=\frac{28}{3}\left(\frac{14}{15}-2\right)
=\frac{28}{3}\left(-\frac{16}{15}\right)
=-\frac{448}{45}.
$$

Therefore

$$
w_2=\frac{14}{15}-0.05\left(-\frac{448}{45}\right)
\approx 1.431111.
$$

At $$w_2\approx 1.431111,$$

$$
L'(w_2)\approx \frac{28}{3}(-0.568889)
\approx -5.309630.
$$

Hence

$$
w_3\approx 1.431111-0.05(-5.309630)
\approx 1.696593.
$$

When $$w<2,$$ the factor $$w-2$$ is negative, so the derivative is negative and the update increases $$w.$$ When $$w>2,$$ it is positive and the update decreases $$w.$$ A much larger learning rate can overshoot repeatedly or diverge.

The data are observed input-target pairs. The model converts an input into a prediction. The loss measures prediction error. The derivative measures local loss sensitivity. The update rule uses that sensitivity to change the parameter.
""")


---

# Module 5 Summary

1. Limits describe nearby behavior and support the definition of continuity.
2. A derivative is the limit of a difference quotient and measures local sensitivity.
3. Differentiation rules make complex formulas manageable.
4. The chain rule multiplies local derivatives through compositions.
5. Backpropagation is an organized reverse application of the chain rule.
6. Critical points are candidates that require classification.
7. Gradient descent moves opposite the derivative, and its learning rate controls stability.

## Cumulative terminology

| AI language | Calculus interpretation |
|---|---|
| Model | Function mapping inputs and parameters to predictions |
| Loss | Objective function to minimize |
| Parameter sensitivity | Derivative of loss with respect to a parameter |
| Backpropagation | Chain-rule evaluation through a computation graph |
| Learning rate | Multiplier controlling update size |
| Training trajectory | Sequence of parameter values generated by updates |
| Convergence | Parameter or loss values approaching stable limits |

## Exit ticket

1. Why is a limit needed in the derivative definition?
2. What is the conceptual difference between a derivative and a learning rate?
3. Why do local derivatives multiply in backpropagation?
4. Why can a zero derivative fail to identify a minimum?
5. What can go wrong when a gradient-descent step is too large?
